In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from pathlib import Path
from scipy.io import mmread
from scipy.stats import entropy
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score


In [ ]:

# ============================================================================
# CONFIGURATION
# ============================================================================

base_dir = Path('./l9_benchmark_sets')
output_csv = "L9_Dataset_Profiling_Metrics.csv"

# Strict Pathing Discovery for L9 Structure: base_dir / Scenario / Replicate
target_dirs = []
if base_dir.exists():
    for s_dir in base_dir.iterdir():
        if s_dir.is_dir() and s_dir.name.startswith('S'):  # Matches S1 to S9
            for rep_dir in s_dir.iterdir():
                if rep_dir.is_dir() and rep_dir.name.startswith('rep'): # Matches rep1, rep2, rep3
                    target_dirs.append(rep_dir)

target_dirs = sorted(target_dirs)
print(f"Found {len(target_dirs)} L9 datasets to profile. Starting sweep...\n")

profiling_results = []


In [ ]:

# ============================================================================
# HELPER FUNCTIONS FOR METRICS
# ============================================================================

def calc_gini(array):
    """Calculate the Gini coefficient of a numpy array."""
    array = np.sort(array.astype(np.float64))
    index = np.arange(1, array.shape[0] + 1)
    n = array.shape[0]
    return (np.sum((2 * index - n - 1) * array)) / (n * np.sum(array))

def calc_knn_purity(adata, label_col='Ground_Truth_Celltype', n_neighbors=15):
    """Calculate the average percentage of neighbors belonging to the same class."""
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep='X_pca')
    # Use connectivities, as it safely represents the exact graph edges
    conn = adata.obsp['connectivities'] 
    
    purities = []
    labels = adata.obs[label_col].values
    
    for i in range(adata.n_obs):
        neighbor_indices = conn[i, :].nonzero()[1]
        if len(neighbor_indices) == 0:
            continue
        
        cell_label = labels[i]
        neighbor_labels = labels[neighbor_indices]
        matches = np.sum(neighbor_labels == cell_label)
        purities.append(matches / len(neighbor_indices))
        
    return np.mean(purities) if purities else np.nan


In [ ]:

# ============================================================================
# MASTER LOOP
# ============================================================================

for target_dir in target_dirs:
    scenario_name = target_dir.parent.name
    rep_name = target_dir.name
    sim_name = f"{scenario_name}_{rep_name}"
    
    print(f"Profiling: {sim_name}...")
    
    try:
        # --- 1. VERIFY FILES ------------------------------------------------
        counts_file = target_dir / 'counts.mtx'
        genes_file = target_dir / 'genes.txt'
        cells_file = target_dir / 'cells.txt'
        meta_file = target_dir / 'metadata.csv'
        
        if not all(f.exists() for f in [counts_file, genes_file, cells_file, meta_file]):
            raise FileNotFoundError("Missing one or more required L9 files.")

        # --- 2. LOAD DATA ---------------------------------------------------
        # Read matrix (transpose to Cells x Genes)
        counts = mmread(counts_file).T.tocsr()
        
        # Read text files (sep='\s+' handles default R write.table spacing)
        genes = pd.read_csv(genes_file, sep=r'\s+', header=None, dtype=str)[0].values
        cells = pd.read_csv(cells_file, sep=r'\s+', header=None, dtype=str)[0].values
        
        # Read CSV metadata (using cell_id as the index to match barcodes)
        metadata = pd.read_csv(meta_file, sep=',', index_col='cell_id')
        metadata.index = metadata.index.astype(str)
        
        # Map the Splatter 'cell_type' column to our standard naming convention
        metadata['Ground_Truth_Celltype'] = metadata['cell_type']
            
        adata = ad.AnnData(X=counts, obs=metadata)
        adata.var_names = genes
        adata.obs_names = cells
        adata.obs_names_make_unique()
        adata.var_names_make_unique()
        
        # Purge any cells with no ground truth label (safety net)
        adata = adata[~adata.obs['Ground_Truth_Celltype'].isna()].copy()
        
        # --- 3. THE BASICS & QC ---------------------------------------------
        n_cells = adata.n_obs
        n_genes = adata.n_vars
        
        sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)
        median_umi = adata.obs['total_counts'].median()
        median_genes = adata.obs['n_genes_by_counts'].median()
        
        total_elements = n_cells * n_genes
        nonzero_elements = adata.X.nnz
        dropout_pct = (1 - (nonzero_elements / total_elements)) * 100
        
        # --- 4. CLASS IMBALANCE ---------------------------------------------
        cluster_counts = adata.obs['Ground_Truth_Celltype'].value_counts()
        n_clusters = len(cluster_counts)
        min_cluster_size = cluster_counts.min()
        max_cluster_size = cluster_counts.max()
        
        p = cluster_counts.values / cluster_counts.sum()
        
        shannon_norm = entropy(p) / np.log(n_clusters) if n_clusters > 1 else 0
        gini_score = calc_gini(cluster_counts.values)
        
        simpson_index = np.sum(p**2)
        simpson_evenness = (1 / simpson_index) / n_clusters if n_clusters > 0 else 0
        
        # --- 5. PREPROCESSING FOR PCA & DISTANCES ---------------------------
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        
        adata_log = adata.copy()
        
        sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat')
        adata = adata[:, adata.var.highly_variable].copy()
        sc.pp.scale(adata, max_value=10)
        
        n_pcs = min(50, adata.n_obs - 1)
        sc.tl.pca(adata, n_comps=n_pcs)
        
        pca_coords = adata.obsm['X_pca']
        labels = adata.obs['Ground_Truth_Celltype'].values
        
        # --- 6. CLUSTER SEPARABILITY METRICS --------------------------------
        if n_clusters > 1:
            sil_score = silhouette_score(pca_coords, labels)
            db_score = davies_bouldin_score(pca_coords, labels)
            ch_raw = calinski_harabasz_score(pca_coords, labels)
            ch_norm = ch_raw / n_cells
        else:
            sil_score, db_score, ch_norm = np.nan, np.nan, np.nan
            
        knn_purity = calc_knn_purity(adata, label_col='Ground_Truth_Celltype')
        
        # --- 7. DIFFERENTIAL EXPRESSION DIFFICULTY --------------------------
        mean_pi_score = np.nan
        
        if n_clusters > 1:
            # Filter out tiny clusters (< 3 cells) that break Wilcoxon math
            valid_groups = cluster_counts[cluster_counts >= 3].index.tolist()
            
            if len(valid_groups) > 1:
                sc.tl.rank_genes_groups(adata_log, 'Ground_Truth_Celltype', groups=valid_groups, method='wilcoxon')
                
                pi_scores = []
                for cluster in adata_log.uns['rank_genes_groups']['names'].dtype.names:
                    pvals = adata_log.uns['rank_genes_groups']['pvals_adj'][cluster][:50]
                    lfcs = adata_log.uns['rank_genes_groups']['logfoldchanges'][cluster][:50]
                    
                    pi = -np.log10(pvals + 1e-300) * np.abs(lfcs)
                    pi_scores.extend(pi)
                    
                if pi_scores:
                    mean_pi_score = np.mean(pi_scores)
        
        bcv_proxy = np.median(adata.var['dispersions_norm'])

        # --- 8. SAVE RESULTS TO DICT ----------------------------------------
        metrics = {
            "Scenario": scenario_name,
            "Replicate": rep_name,
            "Sample": sim_name,
            "Total_Cells": n_cells,
            "Total_Genes": n_genes,
            "Total_CellTypes": n_clusters,
            "Min_Cluster_Size": min_cluster_size,
            "Max_Cluster_Size": max_cluster_size,
            "Dropout_Pct": round(dropout_pct, 2),
            "Median_UMI_Per_Cell": median_umi,
            "Median_Genes_Per_Cell": median_genes,
            "Norm_Shannon_Entropy": round(shannon_norm, 4) if not np.isnan(shannon_norm) else "NaN",
            "Gini_Coefficient": round(gini_score, 4),
            "Norm_Simpson_Evenness": round(simpson_evenness, 4),
            "Silhouette_Score": round(sil_score, 4) if not np.isnan(sil_score) else "NaN",
            "Davies_Bouldin_Index": round(db_score, 4) if not np.isnan(db_score) else "NaN",
            "Calinski_Harabasz_Norm": round(ch_norm, 4) if not np.isnan(ch_norm) else "NaN",
            "KNN_Purity": round(knn_purity, 4) if not np.isnan(knn_purity) else "NaN",
            "Mean_Pi_Score": round(mean_pi_score, 4) if not np.isnan(mean_pi_score) else "NaN",
            "BCV_Dispersion_Proxy": round(bcv_proxy, 4)
        }
        
        profiling_results.append(metrics)
        
        # Clean up memory immediately
        del adata, adata_log, counts, metadata
        gc.collect()
        print("  ✓ Success")

    except Exception as e:
        print(f"  [!] Failed: {str(e)}")


In [ ]:

# ============================================================================
# EXPORT TO CSV
# ============================================================================
print("\n" + "="*70)
if profiling_results:
    df_metrics = pd.DataFrame(profiling_results)
    
    # Sort by Scenario then Replicate for a clean table
    df_metrics.sort_values(by=["Scenario", "Replicate"], inplace=True)
    df_metrics.set_index("Sample", inplace=True)
    
    df_metrics.to_csv(output_csv)
    print(f"Profiling complete! Matrix saved to: {output_csv}")
    print(df_metrics[['Total_Cells', 'Total_CellTypes', 'Silhouette_Score', 'Mean_Pi_Score']].head())
else:
    print("No datasets were successfully processed.")

In [ ]:
===========================================================================
# EXPORT TO CSV
# ============================================================================
print("\n" + "="*70)
if profiling_results:
    df_metrics = pd.DataFrame(profiling_results)
    
    # Sort by Scenario then Replicate for a clean table
    df_metrics.sort_values(by=["Scenario", "Replicate"], inplace=True)
    df_metrics.set_index("Sample", inplace=True)
    
    df_metrics.to_csv(output_csv)
    print(f"Profiling complete! Matrix saved to: {output_csv}")
    print(df_metrics[['Total_Cells', 'Total_CellTypes', 'Silhouette_Score', 'Mean_Pi_Score']].head())
else:
    print("No datasets were successfully processed.")